# Vector Search Refresh — Post-Pipeline Task

Runs after `agents-pipeline` to sync accepted causal chains into the VS index.

1. **INSERT OVERWRITE** `causal_chains_vs` from `news_causal_features` (accepted only)
2. **Trigger** Delta Sync on `causal_chains_idx`
3. **Verify** index status and document count

> Designed to run as a DABs job task dependent on the pipeline task.

In [0]:
# Parameters (overridable via job widgets)
dbutils.widgets.text("catalog", "smart-trading-dev", "Catalog")
dbutils.widgets.text("gold_schema", "03_gold", "Gold Schema")
dbutils.widgets.text("vs_endpoint", "mas-few-shot", "VS Endpoint")
dbutils.widgets.text("vs_index", "smart-trading-dev.03_gold.causal_chains_idx", "VS Index")

CATALOG = dbutils.widgets.get("catalog")
GOLD_SCHEMA = dbutils.widgets.get("gold_schema")
VS_ENDPOINT = dbutils.widgets.get("vs_endpoint")
VS_INDEX = dbutils.widgets.get("vs_index")

SOURCE_TABLE = f"`{CATALOG}`.`{GOLD_SCHEMA}`.news_causal_features"
STAGING_TABLE = f"`{CATALOG}`.`{GOLD_SCHEMA}`.causal_chains_vs"

print(f"Source:  {SOURCE_TABLE}")
print(f"Staging: {STAGING_TABLE}")
print(f"Index:   {VS_INDEX}")

In [0]:
# Refresh CDF-enabled staging table with latest accepted chains.
# INSERT OVERWRITE ensures a clean snapshot each run.
spark.sql(f"""
    INSERT OVERWRITE {STAGING_TABLE}
    SELECT story_id, treatment_class, mechanism, expected_direction,
           confidence, trigger_event, causal_chain_json, validation_status
    FROM {SOURCE_TABLE}
    WHERE validation_status = 'accepted'
""")

staging_count = spark.sql(f"SELECT COUNT(*) AS cnt FROM {STAGING_TABLE}").collect()[0]["cnt"]
print(f"\u2713 Staging table refreshed: {staging_count} accepted chains")

if staging_count == 0:
    dbutils.notebook.exit("WARN: 0 accepted chains — skipping VS sync")

In [0]:
from databricks.vector_search.client import VectorSearchClient
import time

vsc = VectorSearchClient(disable_notice=True)
idx = vsc.get_index(endpoint_name=VS_ENDPOINT, index_name=VS_INDEX)

# Trigger sync
idx.sync()
print(f"\u2713 Sync triggered for {VS_INDEX}")

# Poll until sync completes (timeout 10 min)
MAX_WAIT = 600
POLL_INTERVAL = 15
elapsed = 0

while elapsed < MAX_WAIT:
    status = idx.describe()
    sync_state = status.get("status", {}).get("detailed_state", "UNKNOWN")
    if sync_state in ("ONLINE_NO_PENDING_UPDATE", "ONLINE"):
        break
    print(f"  Sync in progress... ({sync_state}, {elapsed}s elapsed)")
    time.sleep(POLL_INTERVAL)
    elapsed += POLL_INTERVAL
else:
    print(f"\u26a0 Sync did not complete within {MAX_WAIT}s (state: {sync_state})")
    print("  Index will finish syncing in the background.")

# Final status
final = idx.describe()
num_docs = final.get("status", {}).get("num_of_documents", "?")
ready = final.get("status", {}).get("ready", False)

print(f"\n{'='*50}")
print(f"VS INDEX STATUS")
print(f"{'='*50}")
print(f"  Index:     {VS_INDEX}")
print(f"  Endpoint:  {VS_ENDPOINT}")
print(f"  Ready:     {ready}")
print(f"  Documents: {num_docs}")
print(f"  Staging:   {staging_count} accepted chains")

if ready:
    print(f"\n\u2713 VS refresh complete")
    dbutils.notebook.exit(f"OK: {num_docs} documents indexed")
else:
    print(f"\n\u26a0 Index not fully ready — check status manually")
    dbutils.notebook.exit(f"WARN: sync pending, {num_docs} docs so far")